# Firma Digital RSA — Hugo e Hiram

Para que todos nos quede claro esto es una explicacion muuy general

1. Generamos una **llave pública** y una **llave privada**.
2. Convertimos el mensaje a un **hash SHA-256**.
3. Firmamos el hash con la **llave privada**.
4. Verificamos la firma con la **llave pública**.
5. Probamos qué pasa si el mensaje cambia o si se usa una llave incorrecta.


## 1. Importamos librerías permitidas

Usamos solo librerías generales de Python:

- `random`: para generar números aleatorios.
- `hashlib`: para calcular SHA-256.
- `math`: para apoyo matemático básico.

No usamos librerías criptográficas como `rsa`, `cryptography` o `PyCryptodome`.

In [ ]:
# random se usa para generar números aleatorios en la prueba de primalidad y en la generación de primos
import random
# hashlib proporciona la función SHA-256 para convertir el mensaje en un número fijo de bits
import hashlib
# math está disponible por si se necesita alguna operación matemática auxiliar
import math

## 2. Funciones matemáticas básicas

RSA necesita dos operaciones importantes:

- Calcular el **máximo común divisor**.
- Calcular el **inverso modular**.

Una breve explicacion por si alguien ve esto jajaj

- `mcd(e, phi)` revisa que `e` sea compatible con `phi`.
- `inverso_modular(e, phi)` calcula `d`, que será parte de la llave privada.

In [ ]:
# Esta se usa para ver que e y phi si son compatibles, tiene que dar 1, e es el exponente publico. 
# Solo asi podemos calcular el inverso modular que nos da la llave privada
def mcd(a, b):
    # Algoritmo de Euclides: en cada paso reemplaza a con b y b con el residuo de a/b
    # cuando b llega a 0, a contiene el máximo común divisor
    while b != 0:
        a, b = b, a % b
    return a

# Esta funcion la usamos para calcular d , d es la parte de la llave privada (n,d)
def inverso_modular(e, phi):
    # Guardamos el phi original para corregir el resultado si queda negativo al final
    phi_original = phi
    # x0 y x1 llevan el seguimiento de los coeficientes durante el algoritmo extendido de Euclides
    x0, x1 = 0, 1

    # Algoritmo extendido de Euclides: calcula d tal que e*d ≡ 1 mod phi
    while e > 1:
        # cociente entero de la división actual
        cociente = e // phi
        # actualizamos e y phi igual que en el algoritmo de Euclides normal
        e, phi = phi, e % phi
        # actualizamos los coeficientes para rastrear la combinación lineal
        x0, x1 = x1 - cociente * x0, x0

    # si el resultado es negativo lo convertimos a su equivalente positivo mod phi
    if x1 < 0:
        x1 += phi_original

    # x1 es el valor d que satisface e*d ≡ 1 mod phi
    return x1

# Las usamos despues en la generacion de llaves

## 3. Generación de números primos

RSA necesita dos números primos grandes:

- `p`
- `q`

Con ellos se calcula:

```text
n = p * q
```

En este notebook usamos una prueba de primalidad sencilla basada en Miller-Rabin.

In [ ]:
# Checar si el numero es primo en nuestro caso checamos el numero que generamos para ver si si es primo
# Basado en el algoritmo de Miller-Raabin
def es_primo(n, rondas=10):
    # casos base: números menores a 2 no son primos
    if n < 2:
        return False
    # 2 y 3 son primos, los manejamos directamente
    if n in (2, 3):
        return True
    # cualquier par mayor a 2 no es primo
    if n % 2 == 0:
        return False

    # descomponemos n-1 como 2^r * d para poder aplicar Miller-Rabin
    r = 0
    d = n - 1

    # dividimos d entre 2 hasta que sea impar, contando cuántas veces en r
    while d % 2 == 0:
        r += 1
        d //= 2

    # repetimos la prueba con distintas bases aleatorias para aumentar la certeza
    for _ in range(rondas):
        # elegimos una base aleatoria entre 2 y n-2
        a = random.randrange(2, n - 1)
        # calculamos a^d mod n como primer test
        x = pow(a, d, n)

        # si x es 1 o n-1 este testigo no puede demostrar que n es compuesto
        if x == 1 or x == n - 1:
            continue

        # elevamos al cuadrado hasta r-1 veces buscando que x llegue a n-1
        for _ in range(r - 1):
            x = pow(x, 2, n)

            # si encontramos n-1, este testigo no descarta que sea primo
            if x == n - 1:
                break
        else:
            # si nunca llegamos a n-1, n definitivamente es compuesto
            return False

    # si pasó todas las rondas, consideramos que n es primo (probabilísticamente)
    return True

# Esta funcion es para generar los numeros primos muy grandes, o con el parametro que el pasemos (bits)
def primo_aleatorio(bits):
    # intentamos indefinidamente hasta encontrar un primo del tamaño pedido
    while True:
        # generamos un número aleatorio con la cantidad exacta de bits
        candidato = random.getrandbits(bits)

        # forzamos el bit más significativo a 1 para garantizar el tamaño, y el bit 0 a 1 para que sea impar
        candidato = candidato | (1 << (bits - 1)) | 1

        # si pasa la prueba de primalidad, lo retornamos
        if es_primo(candidato):
            return candidato

## 4. Generación de llaves RSA

La función `generar_llaves()` produce:

```text
Llave pública  = (n, e)
Llave privada  = (n, d)
```

La **llave pública** se puede compartir y sirve para verificar.

La **llave privada** debe guardarse en secreto y sirve para firmar.

In [ ]:
def generar_llaves(bits=512):
    print("Generando llaves RSA...")

    # generamos dos primos aleatorios de la mitad del tamaño total de bits pedido
    p = primo_aleatorio(bits // 2)
    q = primo_aleatorio(bits // 2)

    # nos aseguramos de que p y q sean distintos, porque RSA requiere dos primos diferentes
    while p == q:
        q = primo_aleatorio(bits // 2)

    # n es el módulo público, producto de los dos primos secretos
    n = p * q

    # phi(n) es la función de Euler, usada para encontrar e y d
    phi = (p - 1) * (q - 1)

    # 65537 es el exponente público estándar en RSA (primo de Fermat, eficiente y seguro)
    e = 65537

    # si e no es coprimo con phi, buscamos el siguiente impar que sí lo sea
    if mcd(e, phi) != 1:
        e = 3
        while mcd(e, phi) != 1:
            e += 2

    # d es el inverso modular de e respecto a phi, es el exponente privado
    d = inverso_modular(e, phi)

    # empaquetamos las llaves como tuplas (n, exponente)
    llave_publica = (n, e)
    llave_privada = (n, d)

    print("\nValores educativos generados:")
    print(f"p = {p}")
    print(f"q = {q}")
    print(f"n = p*q = {n}")
    print(f"phi(n) = {phi}")
    print(f"e = {e}")
    print(f"d = {d}")

    print("\nLlave pública  (n, e):", llave_publica)
    print("Llave privada  (n, d):", llave_privada)

    print("\nExplicación:")
    print("- La llave pública se puede compartir y se usa para verificar firmas.")
    print("- La llave privada debe mantenerse secreta y se usa para firmar.")

    return llave_publica, llave_privada

## 5. Preparación del mensaje

No firmamos el texto directamente.

Primero calculamos:

```text
SHA-256(mensaje)
```

Ese hash funciona como una huella digital del mensaje.

Si el mensaje cambia aunque sea un carácter, el hash cambia y la firma deja de ser válida.

In [ ]:
def hash_mensaje(mensaje):
    # codificamos el texto a bytes UTF-8 y aplicamos SHA-256 para obtener 32 bytes fijos
    digest = hashlib.sha256(mensaje.encode("utf-8")).digest()
    # convertimos los bytes a hexadecimal para poder mostrarlo legiblemente
    hash_hex = digest.hex()
    # convertimos los bytes a un entero grande en orden big-endian para usarlo en la operación RSA
    hash_entero = int.from_bytes(digest, "big")

    # regresamos ambas representaciones: la visual y la numérica
    return hash_hex, hash_entero


def validar_mensaje(mensaje):
    # verificamos que la entrada sea realmente un string y no otro tipo de dato
    if not isinstance(mensaje, str):
        return False, "El mensaje debe ser texto tipo str."

    # rechazamos mensajes que solo contengan espacios o que estén completamente vacíos
    if mensaje.strip() == "": 
        return False, "El mensaje no puede estar vacío."

    return True, "OK"

## 6. Firma digital

Esta es la parte más importante para la defensa.

La firma se calcula así:

```text
firma = hash^d mod n
```

Donde:

- `hash` es el SHA-256 del mensaje.
- `d` viene de la llave privada.
- `n` es parte de ambas llaves.

En código, la línea central es:

```python
firma = pow(h, d, n)
```

In [ ]:
def firmar(mensaje, llave_privada):
    # validamos el mensaje antes de continuar para evitar errores más adelante
    ok, razon = validar_mensaje(mensaje)

    # si la validación falla, lanzamos un error con la razón específica
    if not ok:
        raise ValueError(razon)

    # extraemos n y d de la llave privada
    n, d = llave_privada

    # obtenemos el hash del mensaje tanto en hex (para mostrarlo) como en entero (para operar)
    hash_hex, h = hash_mensaje(mensaje)

    # el hash debe ser menor que n para que la operación modular tenga sentido
    if h >= n:
        raise ValueError("El hash es mayor o igual que n. Se necesitan llaves más grandes.")

    # esta es la operación central de RSA: firma = hash^d mod n
    firma = pow(h, d, n)

    print("Mensaje:", mensaje)
    print("SHA-256:", hash_hex)
    print("Hash como entero:", h)
    print("\nFirma generada:", firma)

    return firma

# A las otras personas se les tiene que dar la firma el mensaje y la llave publica

## 7. Verificación de firma

Para verificar, usamos la llave pública.

La verificación calcula:

```text
H' = firma^e mod n
```

Luego vuelve a calcular el hash del mensaje recibido.

Si ambos valores son iguales, la firma es válida.

La línea central de verificación es:

```python
h_recuperado = pow(firma, e, n)
```

In [ ]:
def verificar(mensaje, firma, llave_publica):
    # validamos el mensaje recibido antes de cualquier operación
    ok, razon = validar_mensaje(mensaje)

    # si el mensaje no es válido, informamos y retornamos False sin operar
    if not ok:
        print("Mensaje inválido:", razon)
        return False

    # la firma debe ser un entero positivo, cualquier otro tipo es inválido
    if not isinstance(firma, int) or firma < 0:
        print("Firma inválida: debe ser un entero positivo.")
        return False

    # extraemos n y e de la llave pública
    n, e = llave_publica

    # la firma también debe ser menor que n para que la operación tenga sentido
    if firma >= n:
        print("Firma inválida: la firma es mayor o igual que n.")
        return False

    # operación central de verificación: recuperamos el hash original elevando la firma al exponente público
    h_recuperado = pow(firma, e, n)

    # calculamos el hash del mensaje que recibimos para compararlo con el recuperado
    hash_hex, h_mensaje = hash_mensaje(mensaje)

    print("Mensaje recibido:", mensaje)
    print("SHA-256 del mensaje recibido:", hash_hex)
    print("Hash del mensaje:", h_mensaje)
    print("Hash recuperado desde la firma:", h_recuperado)

    # si el hash recuperado desde la firma coincide con el hash del mensaje, la firma es auténtica
    if h_recuperado == h_mensaje:
        print("\nRESULTADO: Firma válida.")
        return True
    else:
        # si no coinciden, el mensaje fue alterado o la firma no corresponde a este mensaje
        print("\nRESULTADO: Firma inválida.")
        return False

## 8. Demo principal

Esta es la parte que puedes usar para grabar el video.

Primero generamos llaves. Luego firmamos un mensaje y lo verificamos.

In [15]:
# generamos el par de llaves RSA de 512 bits para la demo
llave_publica, llave_privada = generar_llaves(bits=512)

Generando llaves RSA...

Valores educativos generados:
p = 93535036432953803871930620116193281877457388251575914887196577996645763896203
q = 61596879740984139767474325709807189470584763965293373082558029763069117607493
n = p*q = 5761466390729225579080686509132970049248223157326341582328876738923687422044535013579884493358104141229980612652552679919113281078703038869427909347049079
phi(n) = 5761466390729225579080686509132970049248223157326341582328876738923687422044379881663710555414464736284154612181204637766896411790733284261668194465545384
e = 65537
d = 1528256278688936897733168351843501401286771005187315898915195892846016481450470724367028461713612996865339331647131565725311308460413314823761232473092161

Llave pública  (n, e): (5761466390729225579080686509132970049248223157326341582328876738923687422044535013579884493358104141229980612652552679919113281078703038869427909347049079, 65537)
Llave privada  (n, d): (57614663907292255790806865091329700492482231573263415823288767389236874

In [16]:
# definimos el mensaje que queremos firmar
mensaje = "Transferencia autorizada: $500 a cuenta 4321"

# firmamos el mensaje con la llave privada y guardamos la firma resultante
firma = firmar(mensaje, llave_privada)

Mensaje: Transferencia autorizada: $500 a cuenta 4321
SHA-256: 5b8bfb87edd268d77b06d447675990797efed1f243c7559bc23ec9a7c35701ba
Hash como entero: 41407796966052851849489093274648887587380340493160617153124276087306791092666

Firma generada: 837193625091252642798035585759253567160919090568491340731000057642897189461021505554913444019157602509443344461887882730383797815212249983413980711354639


In [17]:
# verificamos que la firma corresponde al mensaje usando la llave pública
verificar(mensaje, firma, llave_publica)

Mensaje recibido: Transferencia autorizada: $500 a cuenta 4321
SHA-256 del mensaje recibido: 5b8bfb87edd268d77b06d447675990797efed1f243c7559bc23ec9a7c35701ba
Hash del mensaje: 41407796966052851849489093274648887587380340493160617153124276087306791092666
Hash recuperado desde la firma: 41407796966052851849489093274648887587380340493160617153124276087306791092666

RESULTADO: Firma válida.


True

## 9. Caso: mensaje alterado

Aquí usamos la firma original, pero cambiamos el mensaje.

Esto debe fallar porque el hash del mensaje alterado ya no coincide con el hash recuperado desde la firma.

In [18]:
# cambiamos el monto del mensaje para simular una alteración maliciosa
mensaje_alterado = "Transferencia autorizada: $900 a cuenta 4321"

# intentamos verificar con la firma original, debe fallar porque el hash ya no coincide
verificar(mensaje_alterado, firma, llave_publica)

Mensaje recibido: Transferencia autorizada: $900 a cuenta 4321
SHA-256 del mensaje recibido: 23286345666fad813cef607353ccbe69d0ba066c2c9f3101e7a239cb7f65f200
Hash del mensaje: 15902308726917906578639180018385876614640819531937599168162483581087459439104
Hash recuperado desde la firma: 41407796966052851849489093274648887587380340493160617153124276087306791092666

RESULTADO: Firma inválida.


False

## 10. Caso: llave pública incorrecta

Aquí generamos otro par de llaves y tratamos de verificar con la llave pública equivocada.

Esto debe fallar porque la firma fue creada con otra llave privada.

In [19]:
# generamos un par de llaves completamente diferente para simular una llave equivocada
llave_publica_falsa, llave_privada_falsa = generar_llaves(bits=512)

# intentamos verificar con la llave pública incorrecta, debe fallar
verificar(mensaje, firma, llave_publica_falsa)

Generando llaves RSA...

Valores educativos generados:
p = 86110642991493929405560991908676667962193835710130549155610726991225060713669
q = 82534480540159819959525896943488371605967873269551345546147587690613697081817
n = p*q = 7107097188282105301529056975567162586662697040015418523053769703273068333651082242132504068412842714246058634437762287978779518036050855460467316703256573
phi(n) = 7107097188282105301529056975567162586662697040015418523053769703273068333650913597008972414663477627357206469398194126269799836141349097145785477945461088
e = 65537
d = 3153552744789105729106687471954668172485027235357864574979074766485814534424349106627110148746722147848506402949471065076609842302675309290926372868059393

Llave pública  (n, e): (7107097188282105301529056975567162586662697040015418523053769703273068333651082242132504068412842714246058634437762287978779518036050855460467316703256573, 65537)
Llave privada  (n, d): (71070971882821053015290569755671625866626970400154185230537697032730683

False

## 11. Caso: mensaje vacío

El sistema debe rechazar mensajes vacíos.

In [20]:
# intentamos firmar un mensaje vacío, la validación debe lanzar un ValueError
try:
    firmar("", llave_privada)
except ValueError as error:
    # capturamos el error y lo mostramos para confirmar que el rechazo funciona correctamente
    print("Error capturado correctamente:", error)

# también probamos verificar con mensaje vacío, debe retornar False por la misma validación
verificar("", firma, llave_publica)

Error capturado correctamente: El mensaje no puede estar vacío.
Mensaje inválido: El mensaje no puede estar vacío.


False

## 12. Caso: firma mal formada

Aquí intentamos verificar usando una firma inválida.

In [21]:
# usamos un string en lugar de un entero para simular una firma completamente inválida
firma_mala = "esto no es una firma"

# la verificación debe detectar que el tipo es incorrecto y retornar False de inmediato
verificar(mensaje, firma_mala, llave_publica)

Firma inválida: debe ser un entero positivo.


False